### Import the Data

In [ ]:
%run  Data_preparation_GTEx.ipynb

### Import the Model 

In [ ]:
from Classification_model_GTEx import *

In [ ]:
train_loader = DataLoader(training_set, batch_size=128, shuffle=True)
test_loader = DataLoader(testing_set, batch_size=128, shuffle=False)

In [ ]:
torch.manual_seed(0)

num_hiddens_genotype = 16
num_hiddens_final = 16

model = Drugcell_Vae(term_size_map, term_direct_gene_map, dG, num_genes, 
                 root, num_hiddens_genotype, num_hiddens_final, n_class = len(train_labels))

In [ ]:
def create_term_mask(term_direct_gene_map, gene_dim, device):

    term_mask_map = {}

    for term, gene_set in term_direct_gene_map.items():

        mask = torch.zeros(len(gene_set), gene_dim)

        for i, gene_id in enumerate(gene_set):
            mask[i, gene_id] = 1

        mask_gpu = torch.autograd.Variable(mask)

        term_mask_map[term] = mask_gpu.to(device)

    return term_mask_map

term_mask_map = create_term_mask(model.term_direct_gene_map, num_genes, device = DEVICE)


In [ ]:
model.to(DEVICE)
learning_rate = 0.0001
torch.manual_seed(2025)
loss_list = []
accu_list = []
train_epochs = 50

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import tqdm

In [ ]:
optimizer = Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()  # or FocalLoss if imbalance

term_mask_map = create_term_mask(model.term_direct_gene_map, gene_dim=num_genes, device=DEVICE)

optimizer.zero_grad()

for name, param in model.named_parameters():
    term_name = name.split('_')[0]

    if '_direct_gene_layer.weight' in name:
        param.data = torch.mul(param.data, term_mask_map[term_name].to(DEVICE)) * 1
    else:
        param.data = param.data * 1

for epoch in range(train_epochs):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    train_loop = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{train_epochs}", leave=False)
    
    for i, (data, labels) in enumerate(train_loop):
        # Convert torch tensor to Variable

        # Forward + Backward + Optimize
        optimizer.zero_grad()  # zero the gradient buffer

        # Here term_NN_out_map is a dictionary
        mask_data, mask = model.mask_input(data, 0.2)
        logits, term_NN_out_map, term_original_children, aux_out_map, aux_cancer_map = model(data.to(DEVICE))

        loss_intermidiate = model.intermediate_loss_cancer(aux_cancer_map, labels.to(DEVICE))

        log_loss, class_loss, KLD = model.loss_log_vae(logits, labels.to(DEVICE))

        loss = torch.mean(log_loss + model.inter_loss_penalty * loss_intermidiate)


        loss.backward()
        optimizer.step()
        
        # Stats
        batch_loss = loss.item()
        epoch_loss += batch_loss * data.size(0)

        pred = torch.argmax(logits, dim=1)
        correct += (pred == labels.to(DEVICE)).sum().item()
        total += labels.size(0)
        acc = correct / total

        train_loop.set_postfix(loss=batch_loss, acc=acc)


    epoch_loss /= total
    epoch_acc = correct / total


    
    with torch.no_grad():

        (inputdata, labels) = next(iter(test_loader))

        logits, term_NN_out_map, term_original_children, aux_out_map, aux_cancer_map = model(inputdata.to(DEVICE))

        predictions = torch.argmax(logits, dim=1)  # [batch_size]
        labels = labels.to(DEVICE)

        correct = (predictions == labels.to(DEVICE)).sum().item() 
        total = labels.size(0) 

        accuracy = correct / total
        
        
        loss_list.append(loss)
        accu_list.append(accuracy)
        # if epoch % 10 == 0:
            
    tqdm.tqdm.write(f"[Epoch {epoch+1}] Train Loss: {loss:.4f}, Training accuracy:{epoch_acc:4f} ,Test Accuracy: {accuracy:.4f}")
    

# torch.save(model, "model_gtex_non.pt")





In [ ]:
with open('gtex_loss_list_non.txt', 'w') as f:
    for loss in loss_list:
        f.write(f"{loss}\n")

with open('gtex_accuracy_list_non.txt', 'w') as f:
    for accuracy in accu_list:
        f.write(f"{accuracy}\n")


In [ ]:
plt.plot(accu_list)